# Bài QR (quishing): làm lại từng bước và kiểm mọi con số

Notebook này làm lại toàn bộ phần phân tích của bài QR từ dữ liệu đã công bố, trong một file duy nhất. Mọi hàm đều được viết ngay trong notebook,
nên đọc từ trên xuống là thấy hết cách mỗi con số được tính ra. Notebook chạy độc lập: không cần repo nào khác.

### Cách chạy

1. Tải thư mục chứa notebook này (gồm `qr_walkthrough.ipynb`, thư mục `expected/` và `requirements.txt`).
2. Cài thư viện đúng phiên bản: `pip install -r requirements.txt` (Python 3.12).
3. Tải gói dữ liệu của bài từ Zenodo (DOI ghi trong ô thiết lập bên dưới) vào thư mục `qr_data/` cạnh notebook, hoặc để notebook tự tải.
4. Mở notebook và chạy hết (Run All). Cả notebook chạy khoảng nửa phút sau khi có dữ liệu.

Gói dữ liệu (khoảng 270 MB) gồm:
- bảng kết quả giải mã của sweep 792.000 ảnh với ba thư viện, và của thư viện thứ tư ZXing;
- bảng kết quả giải mã của các ảnh đã phục hồi (bốn file), checkpoint của ba mạng phục hồi, và chỉ mục của tập ảnh phục hồi;
- các sổ thu thập: danh sách trang đã quét, sổ các domain đã nộp lên URLScan, các mã QR tìm được, danh sách tên miền tự sinh, và phán quyết của
  tác giả cho từng trang có mã.

Hai phần không có trong gói, vì chúng cần phần cứng và thời gian mà một notebook không có:
- sinh 792.000 ảnh QR rồi cho các thư viện giải mã (khoảng 8 giờ CPU trên máy có cài WeChatQR); ảnh không cần công bố vì chúng được sinh tất
  định từ mã URL, seed và tham số biến đổi;
- huấn luyện các mạng phục hồi ảnh (cần GPU).

Thư mục `expected/` chứa các kết quả mà bài báo dùng (file JSON).
Notebook chỉ đọc chúng để so; nếu thư mục này không có cạnh notebook, notebook tự tải nó từ repo công khai.

Mỗi bước có ba phần: giải thích bước đó làm gì và vì sao bài cần nó, code làm lại trên dữ liệu, và phần kiểm so kết quả vừa tính với kết quả kỳ
vọng. Cuối mỗi bước có dòng `[step] x/y checks match`. Bước 10 gom tất cả lại. Nhiều phép kiểm so nguyên một file kết quả, từng trường một. Notebook không dùng mạng nơ-ron để tính lại (chỉ đếm tham số trong checkpoint), nên kết quả
giống hệt trên mọi máy.

Thư viện cần có: numpy, pandas, scipy, torch (chỉ để đọc checkpoint ở Bước 8).

### Các bước

| Bước | Làm gì | So với |
|---|---|---|
| 1 | hai nhánh thu thập và dữ liệu thô | |
| 2 | tỷ lệ trang có QR (prevalence), dựng lại đúng bản đã ghim cho bài | `prevalence_snapshot.frozen.json` |
| 3 | bảng giải mã của sweep 792.000 ảnh | |
| 4 | hai phép thử đã đăng ký T1, T2 | `dfr_snapshot.json` |
| 5 | các chẩn đoán đã đăng ký và hai phân tích post-hoc | `dfr_snapshot.json` |
| 6 | thư viện thứ tư ZXing (post-hoc) | `zxing_snapshot.json` |
| 7 | phục hồi ảnh, nghiên cứu thứ nhất | `restore_snapshot.json` |
| 8 | phục hồi ảnh, so sánh ba kiến trúc | `restore_v2_snapshot.json` |
| 9 | chuỗi giải mã có cổng (cách triển khai thật) | |
| 10 | tổng hợp | |

### Thuật ngữ

| Từ | Nghĩa |
|---|---|
| quishing | lừa đảo bằng mã QR: mã dẫn nạn nhân tới trang lừa đảo, hoặc là mã thanh toán chuyển tiền vào tài khoản kẻ gian |
| module | một ô vuông đen/trắng của mã QR; mã càng chứa nhiều ký tự thì càng nhiều module |
| box size | số pixel cho mỗi cạnh module khi vẽ mã (2, 3 hoặc 4 px); 2 px là cỡ đo được trên các trang thật |
| EC level | mức sửa lỗi của mã QR: L (khoảng 7%), M (15%), Q (25%), H (30%) số module hỏng vẫn đọc được |
| decoder | thư viện đọc mã QR: OpenCV, PyZbar, WeChatQR, và ZXing ở Bước 6 |
| transform | một kiểu làm hỏng ảnh: làm mờ (`blur`), mờ chuyển động (`motion`), nhiễu muối tiêu (`saltpepper`), đảo màu (`invert`), phối cảnh (`perspective`), dán logo (`logo`), xoay (`rotate`), giảm tương phản (`contrast`); `clean` là ảnh gốc |
| strength | mức độ làm hỏng: 0,25, 0,50, 0,75, 1,00 |
| DFR | decode failure rate: tỷ lệ ảnh mà decoder không trả ra đúng nội dung mã |
| pp | điểm phần trăm (percentage point): 63,1% trừ 43,4% là 19,7 pp |
| post-hoc | phân tích làm sau khi đã thấy kết quả, không có trong bản đăng ký trước |

## Bước 0: Thiết lập

Ô đầu khai báo đường dẫn tới dữ liệu và kết quả kỳ vọng. Ô thứ hai định nghĩa các hàm dùng để kiểm, giống hệt notebook của bài SMS:
- `check(step, name, mine, saved, tol)` ghi lại một phép so. Với `tol = 0`, hai giá trị phải bằng nhau tuyệt đối.
- `report(step)` in số phép kiểm khớp của một bước, kèm chi tiết từng chỗ lệch nếu có.
- `check_json(step, label, mine, saved_file, skip)` so mọi trường của một kết quả notebook tính ra với một file JSON đã lưu. `skip` là danh
  sách trường cố ý không so, và mỗi lần dùng đều có giải thích lý do.

In [ ]:
import os, csv, json, re, math, hashlib, collections, datetime as dt, warnings
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

import urllib.request

ZENODO_DOI = "10.5281/zenodo.XXXXXXX"   # filled in when the data package is published
ZENODO_RECORD = None                    # e.g. "1234567"; with it set, missing data files are downloaded from Zenodo
EXPECTED_URL = "https://raw.githubusercontent.com/vuthainguyen1602/phishvn/master/notebooks/qr/expected"
DATA_FILES = ["qr_dfr.csv", "qr_dfr_zxing.csv", "qr_restore_arms_dfr.csv", "qr_restore_arms_conv_dfr.csv",
              "qr_restore_arms_hybrid_dfr.csv", "qr_restore_arms_restormer_dfr.csv", "restore.pt", "restore_hybrid.pt",
              "restore_restormer.pt", "qr_restore_v2_index.csv", "qr_scan_examined.csv", "urlscan_qrs.csv",
              "qr_submit_ledger.csv", "qr_submit_qrs.csv", "self_induced_hosts.csv", "qr_adjudication.csv"]
EXPECTED_FILES = ["prevalence_snapshot.frozen.json", "dfr_snapshot.json", "zxing_snapshot.json", "restore_snapshot.json",
                  "restore_v2_snapshot.json"]

DATA = Path.cwd() / "qr_data"
DATA.mkdir(exist_ok=True)
missing = [f for f in DATA_FILES if not (DATA / f).exists()]
if missing and ZENODO_RECORD:
    for f in missing:
        print("downloading", f)
        urllib.request.urlretrieve(f"https://zenodo.org/records/{ZENODO_RECORD}/files/{f}?download=1", DATA / f)
    missing = [f for f in DATA_FILES if not (DATA / f).exists()]
if missing:
    raise SystemExit(f"Missing data files in {DATA}: {missing}. Download the package (DOI {ZENODO_DOI}) into that folder, "
                     "or set ZENODO_RECORD above.")

EXPECTED = Path.cwd() / "expected"
if not all((EXPECTED / f).exists() for f in EXPECTED_FILES):
    EXPECTED = DATA / "expected"
    EXPECTED.mkdir(exist_ok=True)
    for f in EXPECTED_FILES:
        if not (EXPECTED / f).exists():
            urllib.request.urlretrieve(f"{EXPECTED_URL}/{f}", EXPECTED / f)
QR = PAPER = SECTIONS = EXPECTED        # expected results and the two registrations, read only for comparison
saved = lambda name: json.load(open(EXPECTED / name, encoding="utf-8"))
import tempfile
TMP = Path(tempfile.mkdtemp(prefix="qr_walkthrough_"))  # the only place this notebook writes

def read_rows(path):
    if not Path(path).is_file():
        return []
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

print("data:", DATA, "| expected:", EXPECTED)

In [ ]:
CHECKS = []

def _same(a, b):
    return a == b or (isinstance(a, float) and isinstance(b, float) and math.isnan(a) and math.isnan(b))

def check(step, name, mine, ref, tol=0.0):
    ok = abs(float(mine) - float(ref)) <= tol if tol else _same(mine, ref)
    CHECKS.append({"step": step, "check": name, "notebook": mine, "saved": ref, "match": bool(ok)})
    return ok

def report(step):
    rows_ = [c for c in CHECKS if c["step"] == step]
    bad = [c for c in rows_ if not c["match"]]
    print(f"[{step}] {len(rows_) - len(bad)}/{len(rows_)} checks match")
    for c in bad:
        print(f"    MISMATCH {c['check']}: notebook={c['notebook']!r}  saved={c['saved']!r}")

def _plain(o):
    # JSON round trip: tuples become lists, numpy scalars become Python numbers, int keys become strings
    return json.loads(json.dumps(o, default=lambda v: v.item() if hasattr(v, "item") else str(v)))

def flat(d, prefix=""):
    if isinstance(d, dict):
        for k, v in d.items():
            yield from flat(v, f"{prefix}.{k}" if prefix else str(k))
    elif isinstance(d, list) and d and all(isinstance(x, (dict, list)) for x in d):
        for i, v in enumerate(d):
            yield from flat(v, f"{prefix}[{i}]")
    else:
        yield prefix, d

def check_json(step, label, mine, saved_file, skip=()):
    new = dict(flat(_plain(mine)))
    ref = dict(flat(json.load(open(saved_file, encoding="utf-8"))))
    keep = lambda k: not any(k == s or k.startswith(s + ".") for s in skip)
    keys = sorted(k for k in set(new) | set(ref) if keep(k))
    diff = [k for k in keys if not _same(new.get(k, "<missing>"), ref.get(k, "<missing>"))]
    check(step, f"{label}: fields that differ (of {len(keys)})", len(diff), 0)
    for k in diff[:10]:
        print(f"    differs: {k}: saved={ref.get(k, '<missing>')!r} notebook={new.get(k, '<missing>')!r}")


## Bước 1: Hai nhánh thu thập

Câu hỏi đầu tiên của bài: ngoài thực tế, bao nhiêu trang lừa đảo nhắm tới Việt Nam có mã QR? Dữ liệu đến từ hai nhánh, chạy tự động mỗi giờ
(mọi file nhắc tới dưới đây nằm trong gói dữ liệu):

- Nhánh tìm kiếm ("crawled"): tìm trên URLScan các trang đã được người khác gắn thẻ `quishing` hoặc `phishing` + tên miền `.vn`, tải ảnh chụp
  màn hình rồi thử giải mã QR trên ảnh. Mỗi lần quét được ghi vào `qr_scan_examined.csv` (một dòng một lần quét, kèm URL của trang). Mã giải
  được ghi vào `urlscan_qrs.csv`.
- Nhánh nộp ("submitted"): nộp lên URLScan các domain lừa đảo mà repo này tự phát hiện, chờ URLScan chụp ảnh, rồi giải mã QR trên ảnh. Mỗi lần
  nộp được ghi vào `qr_submit_ledger.csv` (cột `shot_file` có giá trị nghĩa là URLScan trả về ảnh chụp). Mã giải được ghi vào `qr_submit_qrs.csv`.

Thêm hai file:
- `self_induced_hosts.csv`: các tên miền "tự sinh", tức chỉ tồn tại vì chính lần nộp trước của repo đã dẫn tới chúng (một dịch vụ đỗ tên miền gắn
  thêm nhãn vào tên vừa được quét). Đây không phải trang ai đó dựng lên, nên bị loại khỏi cả tử số lẫn mẫu số.
- `qr_adjudication.csv`: phán quyết của tác giả cho từng trang có QR mà máy không tự kết luận là vô hại được.

In [ ]:
for name in ("qr_scan_examined.csv", "urlscan_qrs.csv", "qr_submit_ledger.csv", "qr_submit_qrs.csv"):
    rr = read_rows(DATA / name)
    print(f"{name:24} {len(rr):6} rows   columns: {list(rr[0].keys()) if rr else []}")
induced_rows = read_rows(DATA / "self_induced_hosts.csv")
adjudication = read_rows(DATA / "qr_adjudication.csv")
print(f"{'self_induced_hosts.csv':24} {len(induced_rows):6} rows")
print(f"{'qr_adjudication.csv':24} {len(adjudication):6} rows")
pd.DataFrame(adjudication)[["source_page", "verdict", "basis", "adjudicated_on"]]

## Bước 2: Tỷ lệ trang có QR, dựng lại đúng bản đã ghim

### Cách đếm đúng

Prevalence = số trang (không trùng) có ít nhất một mã QR giải được, chia cho số trang đã được xem xét. Bài từng đếm sai ba lần, nên cách đếm
được ghi rõ:
- Tử số và mẫu số đều phải là trang, không phải lần quét. URLScan quét lại cùng một trang nhiều lần; đếm số dòng quét sẽ làm sai tỷ lệ.
- Mẫu số = số trang của nhánh tìm kiếm + số domain của nhánh nộp có ảnh chụp, trừ phần trùng giữa hai nhánh (cùng một tên miền xuất hiện ở cả hai).
  Domain chết khi URLScan ghé thăm (không có ảnh) thì chưa từng được xem xét, nên không vào mẫu số.
- Tên miền tự sinh bị loại khỏi cả hai phía.

### Vì sao phải "ghim"

Hai nhánh vẫn thu thập liên tục, nên mẫu số tăng mỗi giờ. Bài nộp đi phải in đúng con số người phản biện đọc, nên ngày 25/09/2026 kết quả được ghim
vào `prevalence_snapshot.frozen.json`: 27 trang có QR trên 12.843 trang, tức 0,21%. Dữ liệu thô hiện tại đã nhiều hơn thế, nên muốn kiểm bản ghim thì
phải dựng lại trạng thái dữ liệu tại lúc ghim.

Hai file cho phép lọc theo thời gian (`attempted_at` của sổ nộp, `found_at` của mã giải được ở nhánh nộp). File quét của nhánh tìm kiếm thì không có
cột thời gian, nhưng nó chỉ được ghi nối thêm. Vì vậy notebook "neo" vào đúng hai con số của bản ghim:
1. `scans = 946`: lấy 946 dòng đầu của file quét.
2. `submitted_pages = 12.437`: tìm mốc thời gian mà số domain có ảnh chụp tính tới mốc đó (sau khi bỏ tên tự sinh) bằng đúng con số này. Chỉ có
   một mốc thỏa: 00:02:52 ngày 25/09/2026. Lượt nộp tiếp theo lúc 00:50:20, nên dữ liệu được đồng bộ về máy tính giữa hai mốc đó.

Hai con số neo không được tính là phép kiểm. Mọi trường còn lại của bản ghim (số trang của nhánh tìm kiếm, phần trùng, số tên tự sinh bị loại,
số trang có QR, các nhóm phân loại, ba tỷ lệ) là kiểm độc lập.

In [ ]:
PIN = json.load(open(QR / "prevalence_snapshot.frozen.json", encoding="utf-8"))
P = PIN["snapshot"]
print("pinned:", PIN["frozen_on"], "|", PIN["reason"])

examined = read_rows(DATA / "qr_scan_examined.csv")[:P["scans"]]          # anchor 1
ledger = read_rows(DATA / "qr_submit_ledger.csv")
induced = {(r.get("domain") or "").strip().lower() for r in induced_rows}

# anchor 2: the ledger time at which the submitted arm held exactly the pinned number of pages
first_shot = {}
for r in ledger:
    if (r.get("shot_file") or "").strip():
        d = r["domain"].strip().lower()
        first_shot[d] = min(first_shot.get(d, r["attempted_at"]), r["attempted_at"])
cutoffs = [t for t in sorted(set(first_shot.values()))
           if len({d for d, t0 in first_shot.items() if t0 <= t} - induced) == P["submitted_pages"]]
CUT = cutoffs[-1]
next_attempt = min(r["attempted_at"] for r in ledger if r["attempted_at"] > CUT)
print(f"cut-off times reproducing the pinned submitted count: {cutoffs}; next attempt {next_attempt}")

### Phân loại mã QR tìm được

Mỗi mã giải được qua một bước phân loại máy (triage). Đây không phải phán quyết, chỉ ghi lại payload là gì:
- Nếu payload là chuỗi thanh toán EMVCo (chuẩn mà VietQR dùng), hàm `emvco_parse` đọc nó: cấu trúc TLV (2 chữ số tag, 2 chữ số độ dài, rồi giá trị),
  mã ngân hàng (BIN) và số tài khoản nằm trong template 26 đến 51, và tag 63 là checksum CRC-16 để biết chuỗi có hợp lệ không. Ở Việt Nam, mã
  thanh toán chính là payload quishing điển hình: lừa đảo nằm ở chỗ tài khoản nhận tiền không phải tài khoản nạn nhân định trả.
- Nếu là URL, ghi lại tên miền của payload, xem nó có trỏ về chính trang chứa mã không (`same_site`), và nó thuộc loại nào: link mở ứng dụng nhắn
  tin (`messenger`), link rút gọn (`shortener`), không phải URL (`non_url`), hoặc khác (`other`).

Một trang được coi là vô hại tự động (`benign_auto`) khi mọi mã trên trang đều là link nhắn tin hoặc trỏ về chính trang đó. Các trang còn lại do tác giả
đọc tay và xếp vào nhóm theo phán quyết trong `qr_adjudication.csv`.

In [ ]:
TEMPLATE_TAGS = {f"{i:02d}" for i in range(26, 52)}
NAPAS_GUID_HINTS = ("A000000727", "napas", "VIETQR")

def crc16_ccitt(data, poly=0x1021, init=0xFFFF):
    crc = init
    for b in data:
        crc ^= b << 8
        for _ in range(8):
            crc = ((crc << 1) ^ poly) & 0xFFFF if crc & 0x8000 else (crc << 1) & 0xFFFF
    return crc

def parse_tlv(s):
    out, i = {}, 0
    while i + 4 <= len(s):
        tag, ln = s[i:i + 2], s[i + 2:i + 4]
        if not (tag.isdigit() and ln.isdigit()):
            break
        n = int(ln)
        val = s[i + 4:i + 4 + n]
        if len(val) < n:
            break
        out[tag] = val
        i += 4 + n
    return out

def emvco_parse(payload):
    out = {"is_emvco": False, "crc_ok": False, "bin": "", "account": "", "amount": "", "merchant": ""}
    if not payload or parse_tlv(payload).get("00") != "01":
        return out
    out["is_emvco"] = True
    top = parse_tlv(payload)
    idx = payload.rfind("6304")
    if idx != -1 and len(payload) >= idx + 8:
        out["crc_ok"] = payload[idx + 4:idx + 8].upper() == f"{crc16_ccitt(payload[:idx + 4].encode('ascii', 'replace')):04X}"
    out["amount"], out["merchant"] = top.get("54", ""), top.get("59", "")
    for tag in sorted(TEMPLATE_TAGS):
        if tag not in top:
            continue
        for sub in parse_tlv(top[tag]).values():
            deep = parse_tlv(sub)
            if deep.get("00", "").isdigit() and len(deep.get("00", "")) == 6:
                out["bin"], out["account"] = deep["00"], deep.get("01", "")
        if out["bin"]:
            break
    return out

MESSENGER = ("zaloapp.com", "zalo.me", "m.me", "wa.me", "t.me", "messenger.com", "line.me", "api.whatsapp.com")
SHORTENER = ("bit.ly", "tinyurl.com", "goo.gl", "t.co", "is.gd", "cutt.ly", "rb.gy", "shorturl.at", "s.net.vn", "rebrand.ly")

def triage(source_page, payload):
    try:
        ph = (urlparse(payload).hostname or "").lower().removeprefix("www.")
        sh = (urlparse(source_page).hostname or "").lower().removeprefix("www.")
    except ValueError:
        ph = sh = ""
    emv = emvco_parse(payload)
    if emv["is_emvco"]:
        return {"qr_host": "", "same_site": 0, "payload_kind": "emvco_valid" if emv["crc_ok"] else "emvco_badcrc"}
    kind = "other"
    if any(ph == m or ph.endswith("." + m) for m in MESSENGER):
        kind = "messenger"
    elif any(ph == m or ph.endswith("." + m) for m in SHORTENER):
        kind = "shortener"
    elif payload[:5].lower() not in ("http:", "https"):
        kind = "non_url"
    same = int(bool(ph) and bool(sh) and (ph == sh or ph.endswith("." + sh) or sh.endswith("." + ph)))
    return {"qr_host": ph, "same_site": same, "payload_kind": kind}

print(triage("https://www.sinvoice.vn/", "http://zaloapp.com/qr/p/166vwftuwsavo"))
print(triage("https://qq808.me", "https://qq808.me/"))

In [ ]:
VERDICT_BUCKET = {"quishing": "quishing", "phishing_inert_qr": "inert_lure", "abuse_app_download": "abuse",
                  "off_target_app_download": "off_target", "benign": "benign_hand", "benign_payment": "benign_hand"}

def wilson(k, n, z=1.96):
    if not n:
        return (float("nan"), float("nan"))
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (100 * max(0.0, c - h), 100 * min(1.0, c + h))

# denominator
urls = {(r.get("page_url") or "").strip() for r in examined} - {""}
hosts = {(urlparse(u).hostname or "").lower() for u in urls} - {""}
crawled, scans = len(urls), len(examined)
sub = {(r.get("domain") or "").strip().lower() for r in ledger if r["attempted_at"] <= CUT and (r.get("shot_file") or "").strip()}
induced_left_out = len(sub & induced)
sub -= induced
overlap = len(hosts & sub)
pages_examined = crawled + len(sub) - overlap

# numerator: both arms' codes, grouped by page
crawled_hits = read_rows(DATA / "urlscan_qrs.csv")
submit_hits = [r for r in read_rows(DATA / "qr_submit_qrs.csv") if r["found_at"] <= CUT]
by_page, crawled_hit_pages = {}, set()
for r in crawled_hits:
    pg = (r.get("source_page") or "").strip().rstrip("/")
    if pg:
        crawled_hit_pages.add(pg)
        by_page.setdefault(pg, []).append(triage(pg, r.get("qr_decoded_url", "")))
for r in submit_hits:
    pg = (r.get("source_page") or "").strip().rstrip("/")
    if pg and (urlparse(pg).hostname or "").lower() not in induced:
        by_page.setdefault(pg, []).append(triage(pg, r.get("qr_decoded_url", "")))
hits = len(by_page)

verdicts = {(r.get("source_page") or "").strip().rstrip("/"): (r.get("verdict") or "").strip() for r in adjudication}
buckets = {k: 0 for k in ("benign_auto", "benign_hand", "off_target", "abuse", "inert_lure", "quishing", "unadjudicated")}
page_bucket = {}
for pg, found in by_page.items():
    if all(t["payload_kind"] == "messenger" or str(t["same_site"]) == "1" for t in found):
        b = "benign_auto"
    else:
        b = VERDICT_BUCKET.get(verdicts.get(pg, ""), "unadjudicated")
    buckets[b] += 1
    page_bucket[pg] = b

rate = 100 * hits / pages_examined
prev = {"date": CUT[:10], "scans": scans, "crawled_pages": crawled, "submitted_pages": len(sub), "overlap": overlap,
        "pages_examined": pages_examined, "pages_with_qr": hits, "scans_with_qr": len(crawled_hits),
        "crawled_pages_with_qr": len(crawled_hit_pages), "induced_left_out": induced_left_out, "buckets": buckets,
        "prevalence_pct": round(rate, 2), "scan_based_pct": round(100 * len(crawled_hit_pages) / scans, 2),
        "rescan_factor": round(scans / crawled, 2)}
lo, hi = wilson(hits, pages_examined)
print(f"{hits} pages with a code / {pages_examined:,} pages examined = {rate:.2f}%   (Wilson 95% CI {lo:.2f} to {hi:.2f})")
print(f"  {crawled} crawled pages (from {scans} scans) + {len(sub):,} submitted - {overlap} on both; {induced_left_out} self-induced names left out")
print("  buckets:", buckets)
pd.Series(page_bucket).rename("bucket").to_frame()

Ba tỷ lệ "sai cách" được giữ trong kết quả để bài nói rõ vì sao không dùng chúng:
- `scan_based_pct`: số trang có QR của nhánh tìm kiếm chia cho số lần quét (đúng ra phải chia cho số trang);
- `rescan_factor`: trung bình mỗi trang của nhánh tìm kiếm bị quét bao nhiêu lần, tức mức sai lệch nếu nhầm lần quét với trang;
- tỷ lệ theo dòng: số dòng có QR chia cho số lần quét.

Không trang nào thuộc nhóm `quishing` theo nghĩa hẹp (mã đưa nạn nhân tới chỗ bị tấn công). Một trang là "mồi không hoạt động" (`inert_lure`): trang
lừa đảo có hiển thị mã QR nhưng mã đó không dẫn tới đâu.

So với bản ghim. Hai trường không so:
- `adjudicated_on`: lúc ghim, 4 phán quyết đề ngày 25/09; ngày 02/10 tác giả xác nhận lại 4 phán quyết đó từ ảnh chụp và đổi ngày thành 02/10.
  Bản thân phán quyết không đổi (các nhóm phân loại ở trên khớp), chỉ ngày đổi, nên ngày của bản ghim không còn tính lại được từ file hiện tại.
- `tag_quishing`, `tag_vn_phishing`: số trang có thẻ tương ứng trên URLScan, lấy bằng truy vấn trực tiếp tới URLScan (cần khóa API) chứ không từ dữ
  liệu đã thu.

In [ ]:
for k in ("scans", "submitted_pages"):
    print(f"anchor {k}: {prev[k]} (taken from the pin, not a check)")
ref_pin = {k: v for k, v in P.items() if k not in ("scans", "submitted_pages")}
(TMP / "pin.json").write_text(json.dumps(ref_pin))
check_json("2 prevalence", "pinned prevalence snapshot", {k: v for k, v in prev.items() if k not in ("scans", "submitted_pages")},
           TMP / "pin.json", skip=("adjudicated_on", "tag_quishing", "tag_vn_phishing"))
report("2 prevalence")

## Bước 3: Sweep 792.000 ảnh

Không có mẫu quishing thật nào để đo, nên phần đo khả năng đọc mã dùng ảnh tổng hợp. Thiết kế được đăng ký trước (`PREREG_quishing.md`):
- 2.000 URL (phishing và benign), lấy mẫu theo độ dài URL;
- mỗi URL vẽ thành mã QR ở 4 mức sửa lỗi (L, M, Q, H) × 3 cỡ module (2, 3, 4 px);
- mỗi mã được làm hỏng theo 33 biến thể: ảnh gốc, cộng 8 kiểu làm hỏng × 4 mức độ;
- mỗi ảnh cho ba thư viện đọc: OpenCV, PyZbar, WeChatQR.

Tổng cộng 2.000 × 4 × 3 × 33 = 792.000 ảnh, × 3 thư viện = 2.376.000 dòng trong `qr_dfr.csv`. Mỗi dòng ghi thư viện có trả ra gì không (`decoded`)
và có trả đúng nội dung không (`correct`).

Bảng không lưu URL, chỉ lưu 16 ký tự đầu của SHA-1 của URL (phần đầu của `sample_id`). Đó là khóa để gom theo URL, và cũng là lý do bảng có thể
công bố mà không đăng lại corpus.

Đơn vị phân tích là URL, không phải ảnh: 792.000 ảnh là 2.000 URL độc lập nhân với một lưới thiết kế cố định. Mọi đại lượng được tính cho từng URL
trước rồi mới gộp. Nếu gộp theo ảnh, khoảng tin cậy sẽ hẹp đi khoảng 20 lần một cách giả tạo.

In [ ]:
df = pd.read_csv(DATA / "qr_dfr.csv",
                 dtype={"sample_id": "string", "label": "category", "url_len": "int32", "ec_level": "category", "box_size": "int8",
                        "transform": "category", "strength": "category", "qr_version": "int16", "modules": "int16",
                        "decoder": "category", "decoded": "int8", "correct": "int8"},
                 usecols=["sample_id", "label", "url_len", "ec_level", "box_size", "transform", "strength", "qr_version", "modules",
                          "decoder", "decoded", "correct"])
df["url"] = df["sample_id"].str[:16].astype("category")
df.drop(columns=["sample_id"], inplace=True)
df["fail"] = 1 - df["correct"]
n_urls = df["url"].nunique()
decs = sorted(df["decoder"].cat.categories)
expect = n_urls * df["ec_level"].nunique() * df["box_size"].nunique() * 33 * len(decs)
print(f"{len(df):,} rows, {n_urls:,} URLs, decoders {decs}; grid {'complete' if len(df) == expect else 'INCOMPLETE'} ({len(df):,} of {expect:,})")
dfr = {"rows": int(len(df)), "urls": int(n_urls), "decoders": decs, "grid_complete": len(df) == expect}

## Bước 4: Hai phép thử đã đăng ký

T1, cỡ vẽ có thay đổi việc đọc được không? Trên kiểu làm hỏng `rotate`, so DFR ở 2 px với 4 px, ghép cặp theo từng URL (mỗi URL tự so với chính
nó), kiểm định Wilcoxon hai phía. Đăng ký: thành công nếu chênh từ +10 pp trở lên ở ít nhất 2 trong 3 thư viện; một phần nếu từ +5 pp; âm nếu
dưới đó hoặc có thư viện nào ngược dấu. Ý nghĩa: các benchmark thường vẽ mã to; trên trang thật mã nhỏ (khoảng 2 px mỗi module), nên kết luận từ
benchmark có thể không đúng ngoài thực tế.

T2, nhãn phishing/benign có lẫn vào kết quả không? So DFR trung bình theo URL giữa URL phishing và URL benign, không ghép cặp, kiểm định
Mann-Whitney U. Đăng ký: thành công (kết quả mong muốn) nếu chênh dưới 2 pp ở cả ba thư viện; âm nếu từ 5 pp trở lên ở bất kỳ thư viện nào.
Tiêu chí là độ lớn của chênh lệch, không phải p-value, vì "không khác nhau" không chứng minh được bằng việc không bác bỏ.

Hai p-value được hiệu chỉnh Benjamini-Hochberg với m = 2 (số phép thử cố định từ lúc đăng ký).

In [ ]:
from scipy.stats import wilcoxon, mannwhitneyu

def bh(pvals, alpha=0.05):
    items = sorted(pvals.items(), key=lambda kv: kv[1])
    mm = len(items); adj, prev_ = {}, 1.0
    for i in range(mm, 0, -1):
        name, p = items[i - 1]
        prev_ = min(prev_, p * mm / i); adj[name] = prev_
    return {k: {"p": pvals[k], "p_bh": adj[k], "reject": adj[k] <= alpha} for k in pvals}

def per_url_dfr(frame, by=("url", "decoder")):
    return frame.groupby(list(by), observed=True)["fail"].mean().rename("dfr").reset_index()

def t1(frame):
    sub_ = frame[(frame["transform"] == "rotate") & (frame["box_size"].isin([2, 4]))]
    g = sub_.groupby(["url", "decoder", "box_size"], observed=True)["fail"].mean().unstack("box_size")
    out, ps = {}, {}
    for dec, part in g.groupby(level="decoder", observed=True):
        d = (part[2] - part[4]).dropna().to_numpy()
        p = float(wilcoxon(d)[1]) if np.any(d != 0) else 1.0
        out[dec] = {"n_urls": int(d.size), "mean_pp": float(100 * d.mean()), "median_pp": float(100 * np.median(d)),
                    "dfr_box2": float(100 * part[2].mean()), "dfr_box4": float(100 * part[4].mean()), "p": p}
        ps[dec] = p
    means = [v["mean_pp"] for v in out.values()]
    verdict = ("negative" if any(x < 0 for x in means) or sum(x >= 5 for x in means) < 2 else
               "success" if sum(x >= 10 for x in means) >= 2 else "partial")
    return {"per_decoder": out, "p_pooled": float(max(ps.values())), "verdict": verdict}

def t2(frame):
    lab = frame.groupby("url", observed=True)["label"].first()
    d = per_url_dfr(frame).merge(lab.rename("label"), left_on="url", right_index=True)
    out, ps = {}, {}
    for dec, part in d.groupby("decoder", observed=True):
        ph = part.loc[part["label"] == "phishing", "dfr"].to_numpy(); be = part.loc[part["label"] == "benign", "dfr"].to_numpy()
        p = float(mannwhitneyu(ph, be, alternative="two-sided")[1])
        out[dec] = {"n_phishing": int(ph.size), "n_benign": int(be.size), "dfr_phishing": float(100 * ph.mean()),
                    "dfr_benign": float(100 * be.mean()), "diff_pp": float(100 * (ph.mean() - be.mean())), "p": p}
        ps[dec] = p
    diffs = [abs(v["diff_pp"]) for v in out.values()]
    verdict = "negative" if any(x >= 5 for x in diffs) else "success" if all(x < 2 for x in diffs) else "partial"
    return {"per_decoder": out, "p_pooled": float(min(ps.values())), "verdict": verdict}

dfr["T1"], dfr["T2"] = t1(df), t2(df)
dfr["bh"] = bh({"T1": dfr["T1"]["p_pooled"], "T2": dfr["T2"]["p_pooled"]})
display(pd.DataFrame(dfr["T1"]["per_decoder"]).T[["dfr_box2", "dfr_box4", "mean_pp", "median_pp", "p"]].round(4))
print("T1 verdict:", dfr["T1"]["verdict"].upper())
display(pd.DataFrame(dfr["T2"]["per_decoder"]).T[["dfr_phishing", "dfr_benign", "diff_pp", "p"]].round(4))
print("T2 verdict:", dfr["T2"]["verdict"].upper(), "| BH:", {k: round(v["p_bh"], 6) for k, v in dfr["bh"].items()})

T1 đạt: ở 2 px, OpenCV và PyZbar thất bại nhiều hơn hẳn so với 4 px trên ảnh xoay (khoảng +29 và +50 pp), còn WeChatQR gần như không đổi. T2 đạt:
nhãn phishing/benign chỉ làm DFR chênh dưới 1 pp, nên benchmark không bị lẫn với nhãn.

## Bước 5: Các chẩn đoán đã đăng ký và hai phân tích post-hoc

Chẩn đoán đã đăng ký (chỉ mô tả, không có ngưỡng):
- DFR theo thư viện × kiểu làm hỏng, và mức độ mà DFR vượt 50% (nội suy tuyến tính giữa hai mức kề nhau; `<0.25` nếu đã vượt ngay mức nhẹ nhất,
  `none` nếu không bao giờ vượt).
- DFR theo mức sửa lỗi: chuyển từ L lên H mua được bao nhiêu.
- DFR theo cỡ module, cho cả 9 kiểu chứ không chỉ `rotate`.
- Kiểu thất bại: thư viện im lặng (không trả gì) khác với thư viện trả sai nội dung. Im lặng thì hệ thống biết để thử lại; trả sai thì một URL sai
  sẽ đi vào bộ phân loại.
- DFR theo mức độ làm hỏng.

Hai kiểu `blur` và `motion` thất bại gần 100% ở mọi mức độ, vì độ lớn của chúng tính bằng pixel tuyệt đối trong khi mọi trục khác của thiết kế tính
bằng module. Chúng được đánh dấu "bão hòa" và loại khỏi mọi nhận xét về độ dốc theo mức độ, nhưng vẫn nằm trong DFR tổng, vì DFR tổng là thứ một hệ
thống thật gặp phải.

Hai phân tích post-hoc (không có trong bản đăng ký):
- Cascade: chạy cả ba thư viện và lấy kết quả của bất kỳ thư viện nào đọc được thì có hơn dùng thư viện tốt nhất bao nhiêu? Một ảnh được xác định
  bằng ô thiết kế của nó (kiểu, URL, mức sửa lỗi, cỡ, mức độ).
- Mật độ module: số module do độ dài URL quyết định, nên đây là một quan hệ liều-đáp ứng có sẵn trong thiết kế. Dự đoán: dán logo hại mã thưa
  (logo che nhiều module hơn), còn méo hình học hại mã dày (module nhỏ mất lưới nhanh hơn).

In [ ]:
SATURATED = ("blur", "motion")

def crossing(sub_):
    s = sub_.groupby("strength", observed=True)["fail"].mean()
    s = s.reindex(sorted(s.index, key=float)).dropna()
    xs, ys = [float(i) for i in s.index], list(s.to_numpy())
    if not ys:
        return "none"
    if ys[0] >= 0.5:
        return "<%.2f" % xs[0]
    for i in range(1, len(ys)):
        if ys[i] >= 0.5:
            x0, x1, y0, y1 = xs[i - 1], xs[i], ys[i - 1], ys[i]
            return "%.2f" % (x0 + (0.5 - y0) * (x1 - x0) / max(y1 - y0, 1e-9))
    return "none"

tab = df.groupby(["decoder", "transform"], observed=True)["fail"].mean().unstack("decoder")
tab = tab.reindex(sorted(tab.index, key=lambda t: -tab.loc[t].mean()))
dfr["by_transform"] = {tr: {d: float(100 * tab.loc[tr, d]) for d in decs} for tr in tab.index}
dfr["crossing"] = {tr: ("saturated" if tr in SATURATED else
                        {d: crossing(df[(df["transform"] == tr) & (df["decoder"] == d)]) for d in decs}) for tr in tab.index}
ec = df.groupby(["transform", "ec_level"], observed=True)["fail"].mean().unstack("ec_level")
ec = ec[[c for c in ("L", "M", "Q", "H") if c in ec.columns]]
dfr["by_ec"] = {tr: {c: float(100 * ec.loc[tr, c]) for c in ec.columns} for tr in ec.index}
bx = df.groupby(["transform", "box_size"], observed=True)["fail"].mean().unstack("box_size")
dfr["by_box"] = {tr: {int(b): float(100 * bx.loc[tr, b]) for b in bx.columns} for tr in bx.index}
dfr["failure_kind"] = {}
for dec in decs:
    s_ = df[df["decoder"] == dec]
    dfr["failure_kind"][dec] = {"silent_pct": float(100 * (s_["decoded"] == 0).mean()),
                                "wrong_payload_pct": float(100 * ((s_["decoded"] == 1) & (s_["correct"] == 0)).mean()),
                                "correct_pct": float(100 * s_["correct"].mean())}
st = df[df["transform"] != "clean"].groupby(["transform", "strength"], observed=True)["fail"].mean().unstack("strength")
st = st[sorted(st.columns, key=float)]
dfr["by_strength"] = {tr: {c: float(100 * st.loc[tr, c]) for c in st.columns} for tr in st.index}
dfr["saturated"] = list(SATURATED)

print("DFR (%) per decoder x transform, with the strength where DFR crosses 50%:")
display(pd.DataFrame(dfr["by_transform"]).T.round(1).join(pd.Series({k: v if isinstance(v, str) else " / ".join(v.values())
                                                                      for k, v in dfr["crossing"].items()}, name="50% crossing")))
print("what error correction buys (L minus H, pp):", {k: round(v["L"] - v["H"], 1) for k, v in dfr["by_ec"].items()})
print("failure kind:", {k: {kk: round(vv, 4) for kk, vv in v.items()} for k, v in dfr["failure_kind"].items()})

In [ ]:
KEY = ["transform", "url", "ec_level", "box_size", "strength", "modules"]
per = df.groupby(KEY, observed=True)["correct"].max().rename("cascade").reset_index().set_index(KEY)
for d in decs:
    per = per.join(df[df["decoder"] == d].groupby(KEY, observed=True)["correct"].max().rename(d))
per = per.reset_index()
best = max(decs, key=lambda d: per[d].mean())
casc = {}
for tr in sorted(per["transform"].unique()):
    s_ = per[per["transform"] == tr]
    row = {d: float(100 * (1 - s_[d].mean())) for d in decs}
    row["cascade"] = float(100 * (1 - s_["cascade"].mean()))
    row["gain_vs_best_pp"] = row[best] - row["cascade"]
    casc[str(tr)] = row
ob, oc = float(100 * (1 - per[best].mean())), float(100 * (1 - per["cascade"].mean()))
dfr["cascade"] = {"per_transform": casc, "best_decoder": best, "overall_best_dfr": ob, "overall_cascade_dfr": oc,
                  "overall_gain_pp": ob - oc, "registered": False}

bins, names = [0, 25, 33, 41, 49, 10 ** 6], ["<=25", "26-33", "34-41", "42-49", "50+"]
per["density"] = pd.cut(per["modules"].astype(int), bins, labels=names)
dens = {}
for tr in sorted(per["transform"].unique()):
    g = per[per["transform"] == tr].groupby("density", observed=True)[best].mean()
    dens[str(tr)] = {n: (float(100 * (1 - g[n])) if n in g.index else None) for n in names}
counts = per["density"].value_counts().reindex(names)
dfr["by_density"] = {"dfr": dens, "decoder": best, "n_renders": {n: int(counts[n]) for n in names}, "registered": False}

print(f"best single decoder {best}: {ob:.1f}% DFR; all three as a cascade: {oc:.1f}% (gain {ob - oc:.2f} pp)")
display(pd.DataFrame(dens).T.round(1))
check_json("4-5 sweep", "dfr_snapshot.json", dfr, QR / "dfr_snapshot.json")
report("4-5 sweep")

Đọc kết quả: WeChatQR tốt nhất (DFR 43,4%), OpenCV và PyZbar gần nhau (khoảng 63%). Gần như mọi thất bại đều là im lặng: trên 792.000 ảnh mỗi thư
viện, OpenCV trả sai nội dung 2 lần, PyZbar 7 lần, WeChatQR không lần nào (tỷ lệ cao nhất trong ba là 0,0009%). Sửa lỗi chỉ cứu được một kiểu làm hỏng là dán logo (L 81,6% xuống H 5,7%); với các kiểu khác gần như không giúp gì. Cascade
ba thư viện chỉ hơn WeChatQR một chút, phần lớn ở `perspective`. Theo mật độ: logo hại mã thưa nhiều hơn mã dày, còn phối cảnh thì ngược lại, đúng
như dự đoán.

## Bước 6: Thư viện thứ tư ZXing (post-hoc)

ZXing được thêm ngày 03/09/2026, sau khi đã có kết quả của ba thư viện, vì bài gọi ba thư viện kia là "phổ biến" mà bỏ sót thư viện phía máy chủ
người đọc dễ nghĩ tới nhất. Gộp nó vào phân tích đã đăng ký sẽ làm đổi các con số đã cố định trước, nên ZXing được báo cáo riêng như một phân tích
post-hoc, và T1, T2 giữ nguyên.

Lưới giống hệt (cùng seed, cùng 2.000 URL), nên mỗi dòng ở đây khớp với một ảnh của sweep chính. DFR tổng của cả bốn thư viện tính cùng một cách:
mỗi ảnh của lưới tính một lần.

In [ ]:
zx = read_rows(DATA / "qr_dfr_zxing.csv")
by_tr, tot, bad = {}, 0, 0
for r in zx:
    ok = r["correct"] == "1"
    n_, f_ = by_tr.get(r["transform"], (0, 0))
    by_tr[r["transform"]] = (n_ + 1, f_ + (0 if ok else 1))
    tot += 1; bad += 0 if ok else 1
zx_dfr = {t: 100.0 * f_ / n_ for t, (n_, f_) in by_tr.items()}
zx_overall = 100.0 * bad / tot
better = sorted(t for t, v in zx_dfr.items() if t in dfr["by_transform"] and v < min(dfr["by_transform"][t][d] for d in decs))
worse = sorted(t for t, v in zx_dfr.items() if t in dfr["by_transform"] and v > max(dfr["by_transform"][t][d] for d in decs))
reg_overall = {d: 100.0 - dfr["failure_kind"][d]["correct_pct"] for d in decs}
zxing = {"rows": tot, "overall_dfr": round(zx_overall, 2), "by_transform": {t: round(v, 2) for t, v in sorted(zx_dfr.items())},
         "beats_all_three": better, "worse_than_all_three": worse,
         "registered_overall_by_decoder": {d: round(v, 2) for d, v in reg_overall.items()},
         "pooling": "render-weighted over all rows of the grid, ZXing and the registered three alike",
         "note": "post-hoc; the registered tests were computed over three decoders"}
print(f"ZXing: {tot:,} rows, overall DFR {zx_overall:.1f}%; best of the four on {better}, worst on {worse}")
check_json("6 zxing", "zxing_snapshot.json", zxing, QR / "zxing_snapshot.json")

report("6 zxing")

## Bước 7: Phục hồi ảnh, nghiên cứu thứ nhất

Câu hỏi (đăng ký trong `PREREG_restore.md`): một mạng nhỏ làm sạch ảnh trước khi đưa vào thư viện đọc có giúp được không, và có hơn việc đơn giản là
cài WeChatQR không?

Thiết kế: 500 URL, cỡ module 2 px, 4 mức sửa lỗi, 6 kiểu làm hỏng về ảnh (`blur`, `motion`, `saltpepper`, `contrast`, `invert`, `logo`; bỏ xoay và
phối cảnh vì một chồng tích chập không đảo được biến đổi hình học). 25% số URL được giữ riêng để đánh giá, chọn bằng hash của URL. Mạng gồm bốn lớp
tích chập 3×3 (vùng nhìn 9 px, tức 4,5 module), 8 epoch. 4.000 ảnh giữ riêng được đánh giá.

Mỗi ảnh đánh giá có ba phiên bản ("arm"):
- `raw`: ảnh hỏng như nó vốn có;
- `control`: ảnh hỏng sau một dòng tiền xử lý duy nhất: nếu ảnh tối hơn mức trung bình thì đảo màu. Arm này tách phần công của dòng tiền xử lý khỏi
  phần công của mạng;
- `restored`: control rồi qua mạng.

Cả ba arm được ba thư viện đọc trên cùng một máy, lưu trong `qr_restore_arms_dfr.csv`; arm nằm ở phần đuôi `__raw`, `__control`, `__restored` của
`sample_id`.

Hai phép thử đã đăng ký, ghép cặp theo URL, Wilcoxon hai phía, Benjamini-Hochberg với m = 2. D dương khi phía phục hồi tốt hơn:
- T1: restored + OpenCV so với raw + WeChatQR (mạng có hơn việc cài thư viện tốt hơn không?);
- T2: restored + WeChatQR so với raw + WeChatQR (khi đã có thư viện tốt nhất, mạng còn thêm được gì không?).
Ngưỡng: từ +5 pp là thành công, 0 tới +5 là một phần, dưới 0 là âm.

Ngoài ra là các bảng mô tả: DFR theo arm, theo kiểu làm hỏng, số ảnh mạng cứu được và làm hỏng (một con số ròng che hai nhóm khác nhau), và phép
so ghép cặp control với restored cho từng thư viện.

In [ ]:
def load_arms(path):
    rows_ = []
    for r in read_rows(path):
        sid, _, arm = r["sample_id"].rpartition("__")
        assert arm in ("raw", "control", "restored"), r["sample_id"]
        rows_.append({**r, "sample_id": sid, "url_sha1": sid[:16], "arm": arm, "correct": int(r["correct"]), "decoded": int(r["decoded"])})
    return rows_

V1_FILE = "data/processed/qr/qr_restore_arms_dfr.csv"
v1_rows = load_arms(DATA / Path(V1_FILE).name)
v1_decs = sorted({r["decoder"] for r in v1_rows})
ARMS = ("raw", "control", "restored")
agg = {k: collections.defaultdict(lambda: [0, 0]) for k in ("dfr", "tr", "url")}
for r in v1_rows:
    for k, key in (("dfr", (r["arm"], r["decoder"])), ("tr", (r["arm"], r["decoder"], r["transform"])),
                   ("url", (r["arm"], r["decoder"], r["url_sha1"]))):
        agg[k][key][0] += 1; agg[k][key][1] += r["correct"]
F = lambda d, k: 100 * (1 - d[k][1] / d[k][0]) if d[k][0] else float("nan")

restore = {"by_arm": {dn: {arm: F(agg["dfr"], (arm, dn)) for arm in ARMS} for dn in v1_decs}}
trs = sorted({r["transform"] for r in v1_rows})
restore["by_transform"] = {dn: {tr: {"control": F(agg["tr"], ("control", dn, tr)), "restored": F(agg["tr"], ("restored", dn, tr))}
                                for tr in trs} for dn in v1_decs}
idx = {(r["arm"], r["decoder"], r["sample_id"]): r["correct"] for r in v1_rows}
ids = sorted({r["sample_id"] for r in v1_rows})
restore["transitions"] = {}
for dn in v1_decs:
    got = lambda arm, s_: idx.get((arm, dn, s_))
    restore["transitions"][dn] = {"rescued": sum(1 for s_ in ids if not got("control", s_) and got("restored", s_)),
                                  "destroyed": sum(1 for s_ in ids if got("control", s_) and not got("restored", s_)), "n": len(ids)}
r_o, r_w = F(agg["dfr"], ("restored", "opencv")), F(agg["dfr"], ("raw", "wechat"))
restore["vs_wechat"] = {"restored_opencv": r_o, "raw_wechat": r_w, "delta_pp": r_w - r_o}
v1_urls = sorted({r["url_sha1"] for r in v1_rows})
restore["paired"] = {}
for dn in v1_decs:
    d = np.array([F(agg["url"], ("control", dn, u)) - F(agg["url"], ("restored", dn, u)) for u in v1_urls])
    restore["paired"][dn] = {"n_urls": len(v1_urls), "mean_pp": float(d.mean()), "median_pp": float(np.median(d)),
                             "p": float(wilcoxon(d)[1]) if np.any(d != 0) else 1.0}
reg, ps = {}, {}
for name, (lo_arm, hi_arm) in (("T1", (("restored", "opencv"), ("raw", "wechat"))), ("T2", (("restored", "wechat"), ("raw", "wechat")))):
    d = np.array([F(agg["url"], (hi_arm[0], hi_arm[1], u)) - F(agg["url"], (lo_arm[0], lo_arm[1], u)) for u in v1_urls])
    d = d[~np.isnan(d)]
    mean = float(d.mean())
    reg[name] = {"n_urls": int(d.size), "mean_pp": mean, "median_pp": float(np.median(d)),
                 "p": float(wilcoxon(d)[1]) if np.any(d != 0) else 1.0,
                 "verdict": "success" if mean >= 5 else "partial" if mean >= 0 else "negative",
                 "arms": f"{lo_arm[0]}+{lo_arm[1]} vs {hi_arm[0]}+{hi_arm[1]}"}
    ps[name] = reg[name]["p"]
order = sorted(ps, key=lambda k: ps[k]); prev_ = 1.0
for i, k in ((2, order[1]), (1, order[0])):
    prev_ = min(prev_, ps[k] * 2 / i); reg[k]["p_bh"] = prev_
restore["registered"] = reg
restore.update({"source": V1_FILE, "decoders": v1_decs})

display(pd.DataFrame(restore["by_arm"]).T.round(1))
print("transitions control -> restored:", restore["transitions"])
for k, v in reg.items():
    print(f"{k} ({v['arms']}): mean {v['mean_pp']:+.1f} pp, median {v['median_pp']:+.1f} pp, p_BH {v['p_bh']:.3g} -> {v['verdict'].upper()}")
check_json("7 restore v1", "restore_snapshot.json", restore, QR / "restore_snapshot.json")
report("7 restore v1")

Đọc kết quả: chỉ riêng dòng đảo màu đã hạ DFR của OpenCV từ 69,6% xuống 53,5%; mạng hạ tiếp xuống 26,5%. T1 chỉ đạt một phần: restored + OpenCV hơn
raw + WeChatQR khoảng 3,5 pp, dưới ngưỡng 5 pp. T2 đạt: ngay cả khi đã có WeChatQR, thêm mạng vẫn giảm DFR khoảng 13,6 pp.

## Bước 8: Phục hồi ảnh, so sánh ba kiến trúc

Ở nghiên cứu thứ nhất, mạng giúp mọi kiểu làm hỏng cục bộ nhưng không giúp gì với `logo`. Điều đó đã được dự đoán trước từ vùng nhìn 9 px: logo che
một vùng lớn, mạng chỉ nhìn 9 px thì không thể biết bên dưới logo là gì. Nghiên cứu thứ hai (`PREREG_restore_v2.md`) hỏi: một mạng nhìn được cả mã thì
có sửa được `logo` không? Ba kiến trúc, đánh giá trên cùng ảnh giữ riêng, mỗi kiến trúc có đủ ba arm raw, control, restored:
- `conv` (A): mạng của nghiên cứu thứ nhất, được giải mã lại ở đây;
- `hybrid` (B): tích chập rồi self-attention ở 1/4 độ phân giải rồi phóng lên;
- `restormer` (C): attention theo kênh ở độ phân giải đầy đủ.

Phép thử đã đăng ký (D dương khi kiến trúc mới tốt hơn, ghép cặp theo URL, Wilcoxon, Benjamini-Hochberg với m = 4 = 2 phép thử × 2 kiến trúc mới):
- T1 (cơ chế): chỉ ảnh `logo`, OpenCV, restored(mới) so với restored(conv). Thành công từ +10 pp.
- T2 (triển khai): restored(mới) + OpenCV so với raw + WeChatQR trên cả 6 kiểu. Thành công từ +5 pp.

Ràng buộc an toàn (ngưỡng, không phải phép thử): trong các ảnh arm control đã đọc đúng, arm restored không được làm hỏng quá 2%.

Kiểm "đường ống": `raw` trừ `control` chỉ là công của dòng đảo màu, và con số của `conv` ở đây phải tái hiện đúng nghiên cứu thứ nhất (OpenCV 69,6 →
53,5). Nếu lệch, hai lần chạy không so được với nhau.

Cuối cùng là "ngân sách" của mỗi kiến trúc, để một kết quả âm đi kèm cái giá của nó: số tham số (đếm từ checkpoint), kích thước checkpoint, và thời gian
huấn luyện 8 epoch. Thời gian huấn luyện đo lúc chạy ngày 31/08/2026 và được ghi lại dưới dạng hằng số, vì lần huấn luyện đó không lưu kết quả nào khác
mang con số này.

In [ ]:
import torch

V2_FILES = {"conv": "data/processed/qr/qr_restore_arms_conv_dfr.csv", "hybrid": "data/processed/qr/qr_restore_arms_hybrid_dfr.csv",
            "restormer": "data/processed/qr/qr_restore_arms_restormer_dfr.csv"}
TRAIN_SECONDS = {"conv": None, "hybrid": 723, "restormer": 6004}
data_v2 = {k: [{"sid": r["sample_id"], "url": r["url_sha1"], "arm": r["arm"], "decoder": r["decoder"], "transform": r["transform"],
                "correct": r["correct"]} for r in load_arms(DATA / Path(v).name)] for k, v in V2_FILES.items()}
v2_decs = sorted({r["decoder"] for r in data_v2["conv"]})

def per_url(rows_, arm, decoder, transform=""):
    a_ = collections.defaultdict(lambda: [0, 0])
    for r in rows_:
        if r["arm"] == arm and r["decoder"] == decoder and (not transform or r["transform"] == transform):
            a_[r["url"]][0] += 1; a_[r["url"]][1] += r["correct"]
    return {u: 100 * (1 - ok / n_) for u, (n_, ok) in a_.items() if n_}

def paired(new, comp):
    us = sorted(set(new) & set(comp))
    return np.array([comp[u] - new[u] for u in us], dtype=float)

def test(d, bar):
    p = float(wilcoxon(d)[1]) if d.size and np.any(d != 0) else 1.0
    mean = float(d.mean())
    return {"n_urls": int(d.size), "mean_pp": mean, "median_pp": float(np.median(d)), "p": p,
            "verdict": "success" if mean >= bar else "partial" if mean >= 0 else "negative"}

def transitions(rows_, decoder):
    by = {}
    for r in rows_:
        if r["decoder"] == decoder and r["arm"] in ("control", "restored"):
            by.setdefault(r["sid"], {})[r["arm"]] = r["correct"]
    resc = sum(1 for v in by.values() if v.get("control") == 0 and v.get("restored") == 1)
    dest = sum(1 for v in by.values() if v.get("control") == 1 and v.get("restored") == 0)
    ok = sum(1 for v in by.values() if v.get("control") == 1)
    return {"rescued": resc, "destroyed": dest, "control_correct": ok, "n": len(by), "destroyed_pct_of_correct": (100 * dest / ok) if ok else 0.0}

plumb = {arch: {d: {"raw": float(np.mean(list(per_url(rows_, "raw", d).values()))),
                    "control": float(np.mean(list(per_url(rows_, "control", d).values())))} for d in v2_decs}
         for arch, rows_ in data_v2.items()}
for line in plumb.values():
    for v in line.values():
        v["polarity_pp"] = v["raw"] - v["control"]
results_v2, ps2 = {}, {}
base_logo = per_url(data_v2["conv"], "restored", "opencv", "logo")
for arch in ("hybrid", "restormer"):
    rows_ = data_v2[arch]
    results_v2[arch] = {
        "T1": {**test(paired(per_url(rows_, "restored", "opencv", "logo"), base_logo), 10.0),
               "arms": f"restored({arch})+opencv vs restored(conv)+opencv, logo only"},
        "T2": {**test(paired(per_url(rows_, "restored", "opencv"), per_url(rows_, "raw", "wechat")), 5.0),
               "arms": f"restored({arch})+opencv vs raw+wechat"}}
    ps2[f"{arch}:T1"], ps2[f"{arch}:T2"] = results_v2[arch]["T1"]["p"], results_v2[arch]["T2"]["p"]
order = sorted(ps2, key=lambda k: ps2[k]); prev_ = 1.0
for i in range(len(order), 0, -1):
    k = order[i - 1]; prev_ = min(prev_, ps2[k] * 4 / i)
    arch, _, t = k.partition(":"); results_v2[arch][t]["p_bh"] = prev_
guard = {arch: {d: transitions(rows_, d) for d in v2_decs} for arch, rows_ in data_v2.items()}
trs_v2 = sorted({r["transform"] for r in data_v2["conv"]})
per_tr = {}
for arch, rows_ in data_v2.items():
    per_tr[arch] = {}
    for d in v2_decs:
        for t in trs_v2:
            c_, r_ = per_url(rows_, "control", d, t), per_url(rows_, "restored", d, t)
            if c_ and r_:
                per_tr[arch][f"{d}/{t}"] = {"control": float(np.mean(list(c_.values()))), "restored": float(np.mean(list(r_.values())))}
budget = {}
for arch in data_v2:
    ckpt = DATA / ("restore.pt" if arch == "conv" else f"restore_{arch}.pt")
    sd = torch.load(ckpt, map_location="cpu", weights_only=True)
    budget[arch] = {"params": int(sum(v.numel() for v in sd.values())), "checkpoint_kb": round(ckpt.stat().st_size / 1e3, 1),
                    "train_seconds": TRAIN_SECONDS.get(arch)}
restore_v2 = {"registered": results_v2, "guard": guard, "plumbing": plumb, "by_transform": per_tr, "budget": budget,
              "sources": V2_FILES, "m": 4, "bars": {"T1": 10.0, "T2": 5.0}, "guard_pct": 2.0}

print("plumbing (OpenCV raw -> control):", {a: f"{v['opencv']['raw']:.1f} -> {v['opencv']['control']:.1f}" for a, v in plumb.items()})
for arch, v in results_v2.items():
    for t in ("T1", "T2"):
        print(f"{arch:10} {t}: mean {v[t]['mean_pp']:+.1f} pp, p_BH {v[t]['p_bh']:.3g} -> {v[t]['verdict'].upper()}")
print("guard, OpenCV (% of control-correct renders broken):", {a: round(g["opencv"]["destroyed_pct_of_correct"], 2) for a, g in guard.items()})
print("budget:", budget)
check_json("8 restore v2", "restore_v2_snapshot.json", restore_v2, QR / "restore_v2_snapshot.json")
report("8 restore v2")

Đọc kết quả: cả hai kiến trúc mới đều sửa được `logo` (OpenCV giảm khoảng 9,6 đến 9,7 pp so với gần như 0 của `conv`), nhưng dưới ngưỡng 10 pp nên T1
chỉ đạt một phần. Ở T2, `hybrid` hơn raw + WeChatQR khoảng 10,7 pp (thành công), còn `restormer` thua 1,2 pp (âm). Cả hai kiến trúc mới đều vượt ràng
buộc an toàn: chúng làm hỏng khoảng 6,4% và 7,2% số ảnh vốn đọc được, so với 0,5% của `conv`. Cái giá: `hybrid` có số tham số gấp khoảng 6 lần `conv`,
`restormer` gấp khoảng 2,7 lần và huấn luyện mất khoảng 100 phút.

## Bước 9: Chuỗi giải mã có cổng (cách triển khai thật)

Ràng buộc 2% ở Bước 8 đo trên trường hợp "luôn phục hồi", tức đưa mọi ảnh qua mạng. Khi triển khai, không ai làm vậy: chỉ phục hồi khi không đọc được.
Thuật toán 1 của bài là một chuỗi ba tầng:
1. thử đọc ảnh gốc bằng WeChatQR, rồi PyZbar, rồi OpenCV;
2. nếu cả ba đều không trả gì, áp dòng đảo màu rồi thử lại ba thư viện;
3. nếu vẫn không trả gì, đưa qua mạng phục hồi rồi thử lại ba thư viện.

Chuỗi dừng ở thư viện đầu tiên trả ra nội dung, kể cả khi nội dung đó sai, vì lúc triển khai không có đáp án để biết. Trong dữ liệu hiện tại không có
lần dừng sai nào, nhưng phép tính vẫn giữ trường hợp đó để một lần chạy sau không âm thầm biến một nội dung sai thành một lần cứu được. Theo cách này,
mạng chỉ chạm vào những ảnh mà mọi thứ khác đã thất bại, nên không thể làm hỏng ảnh vốn đọc được.

In [ ]:
def gated_metrics(path):
    order_ = ("wechat", "pyzbar", "opencv")
    cells_ = {}
    for row in read_rows(path):
        base, arm = row["sample_id"].rsplit("__", 1)
        cells_.setdefault(base, {}).setdefault(arm, {})[row["decoder"]] = (int(row["decoded"]), int(row["correct"]))
    raw_ok = polarity_added = restore_added = wrong_stop = 0
    for arms in cells_.values():
        stopped = False
        stage_correct = {}
        for arm in ("raw", "control", "restored"):
            stage_correct[arm] = False
            for dec in order_:
                decoded, is_correct = arms[arm][dec]
                if not stopped and decoded:
                    stopped = True
                    stage_correct[arm] = bool(is_correct)
                    wrong_stop += 0 if is_correct else 1
                    break
            if stopped:
                break
        if stage_correct["raw"]:
            raw_ok += 1
        elif stage_correct["control"]:
            polarity_added += 1
        elif stage_correct["restored"]:
            restore_added += 1
    n_ = len(cells_)
    raw_dfr = 100 * (1 - raw_ok / n_)
    gated_dfr = 100 * (1 - (raw_ok + polarity_added + restore_added) / n_)
    return {"n": n_, "raw_dfr": raw_dfr, "gated_dfr": gated_dfr, "gain_pp": raw_dfr - gated_dfr, "raw_ok": raw_ok,
            "polarity_added": polarity_added, "restore_added": restore_added, "wrong_stop": wrong_stop}

gated = {arch: gated_metrics(DATA / Path(path).name) for arch, path in V2_FILES.items()}
pd.DataFrame(gated).T

Đọc kết quả: chuỗi ba thư viện trên ảnh gốc thất bại 29,9%. Thêm dòng đảo màu cứu thêm 31 ảnh, và mạng phục hồi cứu thêm 533 (`conv`), 748 (`hybrid`)
hoặc 413 (`restormer`) ảnh, đưa DFR xuống 15,8%, 10,4% và 18,8%. Vì mạng chỉ chạy khi mọi thứ khác thất bại, việc hai kiến trúc mới vượt ràng buộc an
toàn ở Bước 8 không làm hỏng ảnh nào trong cách triển khai này.

## Bước 10: Tổng hợp

Bảng dưới gom mọi phép kiểm theo bước. Cột `match` là `True` khi kết quả notebook tính ra bằng kết quả đã lưu. Phần lớn phép kiểm so nguyên một file:
bản ghim prevalence, `dfr_snapshot.json`, `zxing_snapshot.json`, `restore_snapshot.json` và `restore_v2_snapshot.json`, từng trường một.

Notebook không làm lại:
- sinh 792.000 ảnh và chạy ba thư viện đọc (notebook bắt đầu từ `qr_dfr.csv`), và chạy ZXing (bắt đầu từ `qr_dfr_zxing.csv`);
- huấn luyện mạng phục hồi và giải mã ảnh đã phục hồi (bắt đầu từ ba arm đã giải mã và checkpoint);
- các hình của bài;
- ở Bước 2: ngày phân xử của bản ghim (bốn phán quyết đã được xác nhận lại và đổi ngày) và hai con số thẻ URLScan (truy vấn trực tiếp URLScan).

Hai con số của bản ghim được dùng làm điểm neo chứ không được kiểm: số lần quét (946) và số domain đã nộp (12.437). Thời gian huấn luyện của hai kiến
trúc mới ở Bước 8 là hằng số đo từ lần huấn luyện.

In [ ]:
tbl = pd.DataFrame(CHECKS)
per_step = tbl.groupby("step", sort=False)["match"].agg(checks="size", matched="sum")
per_step["mismatched"] = per_step["checks"] - per_step["matched"]
display(per_step)
print(f"total: {int(tbl['match'].sum())}/{len(tbl)} checks match")
bad = tbl[~tbl["match"]]
if len(bad):
    display(bad)

In [ ]:
with pd.option_context("display.max_rows", 200):
    display(tbl)